# Host structure and the information in endpoints and time order

Why does a graph add almost nothing on UNSW-NB15? This notebook looks at the structure of the paper's data (the flows of UNSW-NB15_1.csv, 637,891 after duplicate removal) and asks three questions: how many hosts are there and how connected is a graph built on them; does knowing the source and destination IP separate classes that have identical features; and do neighbouring flows in time carry the same label.

In [ ]:
import numpy as np
import pandas as pd

PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"
class_names = ["Analysis", "Backdoor", "DoS", "Exploits", "Fuzzers", "Generic", "Normal", "Reconnaissance", "Shellcode", "Worms"]

df = pd.concat([pd.read_parquet(f"{PATH}/train.parquet"), pd.read_parquet(f"{PATH}/test.parquet")])
df = df[df.index < 700_001].sort_index()
graph = pd.concat([pd.read_csv(f"{PATH}/train_graph.csv", index_col="row_id"),
                   pd.read_csv(f"{PATH}/test_graph.csv", index_col="row_id")]).loc[df.index]
y = df["label"].to_numpy()
feature_cols = [c for c in df.columns if c != "label"]
print(len(df), "flows")

## 1. How many hosts, and how connected is a graph

In [ ]:
endpoint = pd.concat([graph["srcip"].astype(str) + ":" + graph["sport"].fillna(-1).astype(int).astype(str),
                      graph["dstip"].astype(str) + ":" + graph["dsport"].fillna(-1).astype(int).astype(str)])
ips = pd.concat([graph["srcip"], graph["dstip"]])
rows = []
for name, keys in [("(IP, port) endpoints", endpoint), ("IP addresses", ips)]:
    count = keys.value_counts()
    rows.append({"node definition": name, "nodes": len(count), "flows per node (mean)": count.mean(),
                 "flows per node (median)": count.median(), "share of nodes with one flow": (count == 1).mean()})
density = pd.DataFrame(rows)
print(density.round(3).to_string(index=False))
density.to_csv("/kaggle/working/host_graph_density.csv", index=False)

hosts = pd.DataFrame({"y": y, "src": graph["srcip"].to_numpy(), "dst": graph["dstip"].to_numpy()})
per_class = hosts.groupby("y").agg(flows=("y", "size"), source_ips=("src", "nunique"), destination_ips=("dst", "nunique"))
per_class.index = [class_names[i] for i in per_class.index]
print(per_class.to_string())
per_class.to_csv("/kaggle/working/host_per_class.csv")

## 2. Do the IP addresses separate classes that have identical features

Among flows that share an identical feature vector with a flow of another class, the accuracy of predicting the most common label of the group, with and without the (source IP, destination IP) pair added to the group key (in sample).

In [ ]:
key = pd.util.hash_pandas_object(df[feature_cols], index=False).to_numpy()
frame = pd.DataFrame({"k": key, "y": y, "pair": hosts["src"].astype(str).to_numpy() + ">" + hosts["dst"].astype(str).to_numpy()})
ambiguous = frame[frame.groupby("k")["y"].transform("nunique") > 1]


def majority_accuracy(keys):
    table = pd.crosstab(keys, ambiguous["y"])
    return table.max(axis=1).sum() / table.values.sum()


result = pd.DataFrame({"flows": [len(ambiguous)],
                       "accuracy, group of identical features": [majority_accuracy(ambiguous["k"])],
                       "accuracy, group plus IP pair": [majority_accuracy(ambiguous["k"].astype(str) + ambiguous["pair"])]})
print(result.round(3).to_string(index=False))
result.to_csv("/kaggle/working/host_ip_pair_disambiguation.csv", index=False)

## 3. Do neighbouring flows in time carry the same label

For flows of each class, the share whose previous flow from the same source IP (in raw-file order, which is time order) has the same label.

In [ ]:
frame["previous label, same source"] = frame.assign(src=hosts["src"].to_numpy()).groupby("src")["y"].shift(1)
rows = []
for c in range(len(class_names)):
    m = (frame["y"] == c) & frame["previous label, same source"].notna()
    rows.append({"class": class_names[c], "flows": int(m.sum()), "previous flow of the same source has the same label": (frame.loc[m, "previous label, same source"] == c).mean()})
time_order = pd.DataFrame(rows)
print(time_order.round(3).to_string(index=False))
time_order.to_csv("/kaggle/working/host_time_order.csv", index=False)